### Generate & scraping json data

Automating the process of scraping data from Wikipedia and generating data in JSON files because I'm lazy.


#### Imports


In [12]:
import re
import uuid
import json
import requests
import pandas as pd

from io import StringIO
from bs4 import BeautifulSoup
from dotenv import dotenv_values

#### Look-ups


In [13]:
folder_path = "../data/opta/2026 ASEAN Championship/"

with open(folder_path + "groups.json", "r") as f:
    groups_data = json.load(f)
    f.close()

with open(folder_path + "teams.json", "r") as f:
    teams_data = json.load(f)
    f.close()

with open(folder_path + "stages.json", "r") as f:
    stages_data = json.load(f)
    f.close()

In [14]:
groups_lookup = pd.DataFrame(groups_data["groups"])
teams_lookup = pd.DataFrame(teams_data["teams"])
stages_lookup = pd.DataFrame(stages_data["stages"])

#### Create json data


##### Standings


In [9]:
standings = []

for group in groups_lookup["name"]:
    request = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/html/2026_ASEAN_Championship_{group.replace(' ', '_')}",
        headers={
            "User-Agent": f"Match-Analysis-App/1.0 ({dotenv_values('../../.env')['EMAIL']})"
        },
    )

    soup = BeautifulSoup(request.text, "html.parser")
    table = pd.read_html(StringIO(str(soup.find_all("table", {"class": "wikitable"}))))[
        1
    ]

    # Remove host tag from team name
    table["Teamvte"] = table["Teamvte"].str.replace(r"\s*\(H\)", "", regex=True)

    # Add teamId and short team name
    team_ids = []
    short_names = []
    flags = []
    for team in table["Teamvte"]:
        team_info = teams_lookup[
            (teams_lookup["fullName"] == team) | (teams_lookup["shortName"] == team)
        ].reset_index(drop=True)
        # General cases
        if not team_info.empty:
            team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
            short_names.append(
                team_info.loc[0, "shortName"] if len(team_info) > 0 else None
            )
            flags.append(team_info.loc[0, "flag"] if len(team_info) > 0 else None)
        # Special cases
        # else:
        #     # Wikipedia name - FIFA-recognised name
        #     special_cases = {
        #     }
        #     team_info = teams_lookup[
        #         teams_lookup["fullName"] == special_cases.get(team, team)
        #     ].reset_index(drop=True)
        #     team_ids.append(team_info.loc[0, "id"] if len(team_info) > 0 else None)
        #     short_names.append(
        #         team_info.loc[0, "shortName"] if len(team_info) > 0 else None
        #     )
        #     flags.append(team_info.loc[0, "flag"] if len(team_info) > 0 else None)

    # Add teamId column to table
    table["id"] = pd.Series(team_ids)
    table["shortName"] = pd.Series(short_names)
    table["flag"] = pd.Series(flags)

    # Some pre-processing steps
    ## Rename columns
    table.rename(
        columns={
            "Teamvte": "teamName",
            "Pos": "position",
            "Pld": "played",
            "W": "wins",
            "D": "draws",
            "L": "losses",
            "GF": "goalsScored",
            "GA": "goalsConceded",
            "GD": "goalDiff",
            "Pts": "points",
        },
        inplace=True,
    )

    ## Drop unnecessary columns
    table.drop(columns=["Qualification"], inplace=True, errors="ignore")

    ## Move id column in front of teamName
    cols = table.columns.tolist()
    cols.insert(0, cols.pop(cols.index("id")))
    table = table[cols]

    # Convert to dict and add to standings
    standings.append(
        {
            "id": groups_lookup[groups_lookup["name"] == group]["id"].iloc[0],
            "name": group,
            "teams": table.to_dict(orient="records"),
        }
    )

In [11]:
# Write standings to JSON file
with open(folder_path + "standings.json", "w", encoding="utf-8") as f:
    json.dump({"standings": standings}, f, indent=3, ensure_ascii=False)
    f.close()

##### Matches/Results


In [15]:
# Check if the matches file have data
with open(folder_path + "matches.json", "r") as f:
    matches_data = json.load(f)
    f.close()

# Create an ID lookup table
if len(matches_data["matches"]) != 0:
    all_matches_info = [match["matchInfo"] for match in matches_data["matches"]]
    matches_info_lookup = pd.DataFrame(all_matches_info)[
        ["id", "description", "localStartDate", "localStartTime"]
    ]

matches_info_lookup

,id,description,localStartDate,localStartTime
0,e024d91a2e9f47c8,Cambodia vs Singapore,2026-07-24,19:00
1,d9309d84fd0e4a4b,Timor-Leste vs Vietnam,2026-07-24,20:30
2,fd41d5c2ebac4a0a,Singapore vs Timor-Leste,2026-07-27,19:00
3,b5fe9d56546c4c47,Indonesia vs Cambodia,2026-07-27,20:30
4,a0d073b24eed4804,Timor-Leste vs Indonesia,2026-07-31,17:00
5,97556fec73b54b0f,Vietnam vs Singapore,2026-07-31,20:00
6,433954b364894ca9,Cambodia vs Timor-Leste,2026-08-03,17:30
7,a42c7c23d62640d3,Indonesia vs Vietnam,2026-08-03,20:30
8,34c1eca8cdd54080,Vietnam vs Cambodia,2026-08-07,20:00
9,19dc5ffc15384182,Singapore vs Indonesia,2026-08-07,21:00


In [ ]:
matches = []

# For now, just defaulting to group stage
for group in groups_lookup["name"]:
    request = requests.get(
        f"https://en.wikipedia.org/api/rest_v1/page/html/2026_ASEAN_Championship_{group.replace(' ', '_')}",
        headers={
            "User-Agent": f"Match-Analysis-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
        },
    )

    soup = BeautifulSoup(request.text, "html.parser")
    all_matches = soup.find_all("section")[3].find_all("section")

    for match in all_matches:
        # Match info
        match_description = match.find("h3").text.strip()
        match_date = (
            match.find("time")
            .find_all("span", {"class": "bday dtstart published updated itvstart"})[0]
            .text.strip()
        )

        ## Fill in what's available
        match_info = {
            "id": (
                (uuid.uuid4().hex)[:16]
                if (len(matches_data["matches"]) == 0)
                or (
                    matches_info_lookup.loc[
                        (matches_info_lookup["description"] == match_description)
                        & (matches_info_lookup["localStartDate"] == match_date)
                    ].empty
                )
                else (
                    matches_info_lookup.loc[
                        (matches_info_lookup["description"] == match_description)
                        & (matches_info_lookup["localStartDate"] == match_date),
                        "id",
                    ].values[0]
                )
            ),
            "description": match_description,
            "localStartDate": match_date,
            "localStartTime": "",
            "stage": {
                "id": stages_lookup[stages_lookup["name"] == "Group Stage"]["id"].iloc[
                    0
                ],
                "name": "Group Stage",
                "group": {
                    "id": groups_lookup[groups_lookup["name"] == group]["id"].iloc[0],
                    "name": group,
                },
            },
            "contestants": [
                {
                    "id": "",
                    "name": "",
                    "position": "home",
                },
                {
                    "id": "",
                    "name": "",
                    "position": "away",
                },
            ],
            "venue": re.sub(
                r"(\[[1-9]\])+",
                "",
                match.find("span", {"itemprop": "name address"}).text.strip(),
            ),
        }

        ## Extract the time string and clean it
        time_str = (
            match.find("time")
            .find("div", {"class": "ftime"})
            .text.replace("\xa0", " ")
            .replace(".", "")
            .strip()
        )

        ## Remove timezone if found
        if "UTC" in time_str:
            time_str = time_str.rsplit(" ", 1)[0].strip()

        ## Convert to 24-hour
        parsed_time = pd.to_datetime(time_str, format="%I:%M %p", errors="coerce")
        if pd.isna(parsed_time):
            parsed_time = pd.to_datetime(time_str, format="%H:%M", errors="coerce")

        time_24h = parsed_time.strftime("%H:%M") if not pd.isna(parsed_time) else ""
        match_info["localStartTime"] = time_24h

        ## Get the contestants
        home_team = match.find("h3").text.strip().split(" vs ")[0]
        away_team = match.find("h3").text.strip().split(" vs ")[1]

        match_info["contestants"][0]["name"] = home_team
        match_info["contestants"][1]["name"] = away_team

        ## Get contestant IDs
        ### General cases
        if (
            not teams_lookup[
                (teams_lookup["fullName"] == home_team)
                | (teams_lookup["shortName"] == home_team)
            ].empty
            and not teams_lookup[
                (teams_lookup["fullName"] == away_team)
                | (teams_lookup["shortName"] == away_team)
            ].empty
        ):
            home_team_info = teams_lookup[
                (teams_lookup["fullName"] == home_team)
                | (teams_lookup["shortName"] == home_team)
            ].reset_index(drop=True)
            away_team_info = teams_lookup[
                (teams_lookup["fullName"] == away_team)
                | (teams_lookup["shortName"] == away_team)
            ].reset_index(drop=True)
        ### Special cases
        # else:
        #     # Wikipedia name - FIFA-recognised name
        #     special_cases = {}
        #     home_team_info = teams_lookup[
        #         teams_lookup["fullName"]
        #         == special_cases.get(home_team, home_team) | teams_lookup["shortName"]
        #         == special_cases.get(home_team, home_team)
        #     ].reset_index(drop=True)
        #     away_team_info = teams_lookup[
        #         teams_lookup["fullName"]
        #         == special_cases.get(away_team, away_team) | teams_lookup["shortName"]
        #         == special_cases.get(away_team, away_team)
        #     ].reset_index(drop=True)

        match_info["contestants"][0]["id"] = (
            home_team_info.loc[0, "id"] if len(home_team_info) > 0 else None
        )
        match_info["contestants"][1]["id"] = (
            away_team_info.loc[0, "id"] if len(away_team_info) > 0 else None
        )

        # -------------------------------------------------------------------------
        # Match data
        match_data = {
            "matchStatus": (
                "Fixture"
                if "v" in match.find("th", {"class": "fscore"}).text.strip()
                else "Played"
            ),
            "matchLengthMin": "",
            "matchLengthSec": "",
            "period": [
                {
                    "id": 1,
                    "lengthMin": "",
                    "lengthSec": "",
                    "stoppageTime": "",  # In seconds
                },
                {
                    "id": 2,
                    "lengthMin": "",
                    "lengthSec": "",
                    "stoppageTime": "",  # In seconds
                },
            ],
            "scores": {
                "ht": {
                    "home": 0,
                    "away": 0,
                },
                "ft": {
                    "home": 0,
                    "away": 0,
                },
                "et": {
                    "home": 0,
                    "away": 0,
                },
                "total": {
                    "home": (
                        len(
                            match.find("td", {"class": "fhgoal"})
                            .text.strip()
                            .split("\n")
                        )
                        if match.find("td", {"class": "fhgoal"}).text != ""
                        else 0
                    ),
                    "away": (
                        len(
                            match.find("td", {"class": "fagoal"})
                            .text.strip()
                            .split("\n")
                        )
                        if match.find("td", {"class": "fagoal"}).text != ""
                        else 0
                    ),
                },
            },
        }

        # -------------------------------------------------------------------------
        # Combine match info and data, and add to matches list
        matches.append({"matchInfo": match_info, "matchData": match_data})

In [17]:
# Write matches to JSON file
with open(folder_path + "matches.json", "w", encoding="utf-8") as f:
    json.dump({"matches": matches}, f, indent=3, ensure_ascii=False)
    f.close()

In [ ]:
request = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/html/2026_ASEAN_Championship_knockout_stage",
    headers={
        "User-Agent": f"Match-Analysis-App/1.0 ({dotenv_values('../.env')['EMAIL']})"
    },
)

soup = BeautifulSoup(request.text, "html.parser")
all_sections = soup.find_all("section")
knockout_sections = []

for section in all_sections:
    section_header = section.find("h2")
    if (section_header is not None) and (
        section_header.text.strip() in stages_lookup["name"].values
    ):
        knockout_sections.append(section)

In [64]:
for knockout_stage in knockout_sections:
    stage_name = knockout_stage.find("h2").text.strip()
    stage_id = stages_lookup[stages_lookup["name"] == stage_name]["id"].iloc[0]

    all_matches = knockout_stage.find_all("section")
    match_leg = ""

    for match in all_matches:
        # Match info
        if (match.find("h3") is not None) and (
            "leg" in match.find("h3").text.strip().lower()
        ):
            match_leg = match.find("h3").text.strip()
            continue

        match_description = (
            match.find("h3").text.strip()
            if match_leg == ""
            else f"{match.find('h4').text.strip()} ({match_leg})"
        )
        match_date = (
            match.find("time")
            .find_all("span", {"class": "bday dtstart published updated itvstart"})[0]
            .text.strip()
        )

        ## Fill in what's available
        match_info = {
            "id": (
                (uuid.uuid4().hex)[:16]
                if (len(matches_data["matches"]) == 0)
                or (
                    matches_info_lookup.loc[
                        (matches_info_lookup["description"] == match_description)
                        & (matches_info_lookup["localStartDate"] == match_date)
                    ].empty
                )
                else (
                    matches_info_lookup.loc[
                        (matches_info_lookup["description"] == match_description)
                        & (matches_info_lookup["localStartDate"] == match_date),
                        "id",
                    ].values[0]
                )
            ),
            "description": match_description,
            "localStartDate": match_date,
            "localStartTime": "",
            "stage": {
                "id": stage_id,
                "name": stage_name,
                "group": None,
            },
            "contestants": [
                {
                    "id": "",
                    "name": "",
                    "position": "home",
                },
                {
                    "id": "",
                    "name": "",
                    "position": "away",
                },
            ],
            "venue": (
                re.sub(
                    r"(\[[1-9]\])+",
                    "",
                    match.find("span", {"itemprop": "name address"}).text.strip(),
                )
                if match.find("span", {"itemprop": "name address"}) is not None
                else ""
            ),
        }

        print(match_info)

{'id': '851b0f7ef9264647', 'description': 'Runner-up Group A vs Winner Group B (First leg)', 'localStartDate': '2026-08-15', 'localStartTime': '', 'stage': {'id': 'f7g8h9i0j1k2l3m4', 'name': 'Semi-finals', 'group': None}, 'contestants': [{'id': '', 'name': '', 'position': 'home'}, {'id': '', 'name': '', 'position': 'away'}], 'venue': ''}
{'id': 'a7b89559fd7c466c', 'description': 'Runner-up Group B vs Winner Group A (First leg)', 'localStartDate': '2026-08-16', 'localStartTime': '', 'stage': {'id': 'f7g8h9i0j1k2l3m4', 'name': 'Semi-finals', 'group': None}, 'contestants': [{'id': '', 'name': '', 'position': 'home'}, {'id': '', 'name': '', 'position': 'away'}], 'venue': ''}
{'id': '025ad8ea40114ec2', 'description': 'Winner Group B vs Runner-up Group A (Second leg)', 'localStartDate': '2026-08-18', 'localStartTime': '', 'stage': {'id': 'f7g8h9i0j1k2l3m4', 'name': 'Semi-finals', 'group': None}, 'contestants': [{'id': '', 'name': '', 'position': 'home'}, {'id': '', 'name': '', 'position': 'a

In [69]:
knockout_sections[1]

<section data-mw-section-id="11" id="mwSg"><h2 id="Final">Final</h2>
<span about="#mwt34" class="mw-empty-elt" data-mw='{"parts":[{"template":{"target":{"wt":"main","href":"./Template:Main"},"params":{"1":{"wt":"2026 ASEAN Championship final"}},"i":0}}]}' id="mwSw" typeof="mw:Transclusion"><style about="#mwt35" data-mw='{"name":"templatestyles","attrs":{"src":"Module:Hatnote/styles.css"},"body":{"extsrc":""}}' data-mw-deduplicate="TemplateStyles:r1353705441" typeof="mw:Extension/templatestyles">.mw-parser-output .hatnote{font-style:italic}.mw-parser-output div.hatnote{padding-left:1.6em;margin-bottom:0.5em}.mw-parser-output .hatnote i{font-style:normal}.mw-parser-output .hatnote+span.mw-empty-elt+.hatnote,.mw-parser-output .hatnote+link+.hatnote{margin-top:-0.5em}@media print{body.ns-0 .mw-parser-output .hatnote{display:none!important}}</style></span><div about="#mwt34" class="hatnote navigation-not-searchable" id="mwTA" role="note">Main article: <a href="./2026_ASEAN_Championship_fina

##### Squads


In [15]:
squad_request = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/html/2026_ASEAN_Championship_squads",
    headers={
        "User-Agent": f"Match-Analysis-App/1.0 ({dotenv_values('../../.env')['EMAIL']})"
    },
)

squad_soup = BeautifulSoup(squad_request.text, "html.parser")

In [38]:
all_teams = squad_soup.find_all("h3")[:10]
all_squads = squad_soup.find_all("table", {"class": "wikitable"})

In [ ]:
squad = []

for i in range(len(all_teams)):
    team_name = all_teams[i].text.strip()

    # Find team ID
    if (team_name in teams_lookup["fullName"].values) | (
        team_name in teams_lookup["shortName"].values
    ):
        team_id = teams_lookup[
            (teams_lookup["fullName"] == team_name)
            | (teams_lookup["shortName"] == team_name)
        ]["id"].iloc[0]
    # else:
    #     # Handle special cases where Wikipedia name differs from FIFA-recognised name
    #     special_cases = {}
    #     team_id = teams_lookup[
    #         teams_lookup["fullName"] == special_cases.get(team_name, team_name)
    #     ]["id"].iloc[0]

    # Extract squad table
    squad_table = pd.read_html(StringIO(str(all_squads[i])))[0]

    if (len(squad_table) != 0) and ("Player" in squad_table.columns):
        # Retain relevant columns
        squad_table = squad_table[["No.", "Player", "Pos.", "Club"]]

        squad_table.rename(
            columns={
                "No.": "shirtNumber",
                "Player": "playerName",
                "Pos.": "position",
                "Club": "parentClub",
            },
            inplace=True,
        )

        # Drop rows with NaN values in playerName column
        squad_table.dropna(subset=["playerName"], inplace=True)

        # Fill in missing shirt numbers with 0
        squad_table["shirtNumber"] = squad_table["shirtNumber"].fillna(0)

        # Add player IDs
        player_ids = []
        for player in squad_table["playerName"]:
            player_ids.append((uuid.uuid4().hex)[:16])

            # Remove captain tag if found
            if player.endswith(" (captain)"):
                squad_table.loc[squad_table["playerName"] == player, "playerName"] = (
                    player.replace(" (captain)", "")
                )
        squad_table["id"] = pd.Series(player_ids)

        # Move id column to the front
        cols = squad_table.columns.tolist()
        cols.insert(0, cols.pop(cols.index("id")))
        squad_table = squad_table[cols]

        # Fill in missing player IDs with new UUIDs
        squad_table["id"] = squad_table["id"].apply(
            lambda x: (uuid.uuid4().hex)[:16] if pd.isna(x) else x
        )

        # Add team squad to squads list
        squad.append(
            {
                "teamId": team_id,
                "teamName": team_name,
                "players": squad_table.to_dict(orient="records"),
            }
        )

In [44]:
# Write squads to JSON file
with open(folder_path + "squads.json", "w", encoding="utf-8") as f:
    json.dump({"squads": squad}, f, indent=3, ensure_ascii=False)
    f.close()